# Korea Pine-Wilt Fisher-KPP Simulation

This notebook loads the compact Korea Forest Service pine-wilt observation data committed in this repository, builds yearly density grids, and runs a 2D Fisher-KPP RK4 forward simulation from the 2016 observed density. The raw annual CSV files are tracked by checksum in the manifest because several source files exceed normal GitHub blob limits.

## 1. Setup

In Colab, this cell clones or refreshes the GitHub repository under `/content/fisher-pinn`. Locally, run the notebook from the repository root or any child directory.

In [ ]:
%matplotlib inline

from __future__ import annotations

import csv
import json
import subprocess
import sys
from argparse import Namespace
from pathlib import Path

from IPython.display import Image, Markdown, display

REPO_URL = "https://github.com/rladbsco24/fisher-pinn.git"
REPO_BRANCH = "main"


def _has_project(root: Path) -> bool:
    return (
        (root / "fisher_origin_lab").exists()
        and (root / "data" / "korea_pine_wilt" / "processed" / "manifest.json").exists()
    )


def _run_git(args: list[str], cwd: Path | None = None) -> None:
    subprocess.run(["git", *args], cwd=str(cwd) if cwd else None, check=True)


def _prepare_colab_repo(repo_dir: Path) -> Path:
    if not repo_dir.exists():
        _run_git(["clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(repo_dir)])
    elif (repo_dir / ".git").exists():
        _run_git(["fetch", "--depth", "1", "origin", REPO_BRANCH], cwd=repo_dir)
        _run_git(["checkout", "--force", "FETCH_HEAD"], cwd=repo_dir)
    return repo_dir.resolve()


PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if _has_project(candidate):
        PROJECT_ROOT = candidate
        break

if not _has_project(PROJECT_ROOT) and Path("/content").exists():
    PROJECT_ROOT = _prepare_colab_repo(Path("/content/fisher-pinn"))

if not _has_project(PROJECT_ROOT):
    raise RuntimeError(
        "Project files were not found. Run this notebook from the fisher-pinn repository root "
        "or use Colab with network access so the repository can be cloned."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"project root: {PROJECT_ROOT}")


## 2. Configuration

The defaults are intended to run in a few minutes on a typical Colab or local CPU session. Increase `GRID_SIZE` and `STEPS_PER_YEAR` for a finer RK4 baseline.

In [ ]:
from fisher_origin_lab.korea_data import load_manifest, load_korea_pine_wilt_points
from scripts.run_korea_pine_wilt_simulation import run as run_korea_pine_wilt_simulation

OUT_DIR = PROJECT_ROOT / "runs" / "korea_pine_wilt_notebook"
GRID_SIZE = 96
DIFFUSION = 0.0015
REACTION = 0.70
STEPS_PER_YEAR = 80
END_YEAR = 2030

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"output dir: {OUT_DIR}")

## 3. Dataset Manifest And Integrity

The compact files contain infected-tree coordinates and observation year only. The manifest records raw CSV sizes and sha256 checksums so the source bundle can be audited separately.

In [ ]:
manifest = load_manifest()
compact = manifest["compact_files"]
raw_by_year = {entry["year"]: entry for entry in manifest["raw_files"]}

data_root = PROJECT_ROOT / "data" / "korea_pine_wilt"
csv_gz_path = data_root / compact["csv_gzip"]
npz_path = data_root / compact["npz"]
for data_path in [csv_gz_path, npz_path]:
    if not data_path.exists():
        raise FileNotFoundError(f"Required compact data file is missing: {data_path}")

expected_records = sum(int(value) for value in compact["year_counts"].values())
if int(compact["records"]) != expected_records:
    raise ValueError("Manifest record count does not match year_counts sum.")

rows = ["| year | compact records | raw CSV MB | raw sha256 prefix |", "|---:|---:|---:|---|"]
for year, count in compact["year_counts"].items():
    raw = raw_by_year[int(year)]
    rows.append(f"| {year} | {int(count):,} | {raw['bytes'] / 1_000_000:.1f} | `{raw['sha256'][:12]}` |")

display(Markdown("\n".join(rows)))
print(json.dumps(manifest["source"], indent=2, ensure_ascii=False))
print(f"compact csv.gz: {csv_gz_path} ({csv_gz_path.stat().st_size / 1_000_000:.1f} MB)")
print(f"compact npz:    {npz_path} ({npz_path.stat().st_size / 1_000_000:.1f} MB)")


## 4. Load Compact Observation Points

The coordinate reference system is `EPSG:5179`. This step loads 3.18 million compact observation points into memory from the committed NPZ file.

In [ ]:
points = load_korea_pine_wilt_points()
print(f"records: {len(points.year):,}")
print(f"years: {int(points.year.min())}..{int(points.year.max())}")
print(f"x range: {points.x.min():.1f}..{points.x.max():.1f}")
print(f"y range: {points.y.min():.1f}..{points.y.max():.1f}")
print(f"crs: {points.crs}")

## 5. Run 2D Fisher-KPP RK4 Simulation

The 2016 observed density grid is used as the initial condition. The simulation uses a normalized 2D Fisher-KPP equation with no-flux boundary handling.

In [ ]:
summary = run_korea_pine_wilt_simulation(
    Namespace(
        grid_size=GRID_SIZE,
        pad_m=15_000.0,
        capacity_percentile=99.0,
        smooth_passes=1,
        diffusion=DIFFUSION,
        reaction=REACTION,
        steps_per_year=STEPS_PER_YEAR,
        end_year=END_YEAR,
        output_dir=OUT_DIR,
    )
)
print(json.dumps(summary, indent=2, ensure_ascii=False))

## 6. Observed-Year Metrics

The table compares the RK4 simulation against observed density grids for 2016-2023. This is a forward PDE baseline and not a calibrated real-data epidemiological model.

In [ ]:
metrics_path = OUT_DIR / "korea_pine_wilt_metrics.csv"
with metrics_path.open("r", encoding="utf-8", newline="") as f:
    metric_rows = list(csv.DictReader(f))

md_rows = ["| year | relative L2 | correlation | observed mean | simulated mean |", "|---:|---:|---:|---:|---:|"]
for row in metric_rows:
    md_rows.append(
        "| {year} | {l2:.4f} | {corr:.4f} | {obs:.4f} | {sim:.4f} |".format(
            year=row["year"],
            l2=float(row["relative_l2"]),
            corr=float(row["correlation"]),
            obs=float(row["observed_mean"]),
            sim=float(row["simulated_mean"]),
        )
    )

display(Markdown("\n".join(md_rows)))

## 7. Visual Outputs

The script writes the same PNG files displayed below into `runs/korea_pine_wilt_notebook`. These figures show gridded observations, the RK4 forecast timeline, and observed-year metric trajectories.

In [ ]:
for image_name in [
    "observed_density_by_year.png",
    "rk4_forecast_timeline.png",
    "observed_vs_simulated_metrics.png",
]:
    image_path = OUT_DIR / image_name
    display(Markdown(f"### `{image_name}`"))
    display(Image(filename=str(image_path)))

## 8. Notes

This notebook is a reproducible bridge from the Korea Forest Service observation coordinates to a Fisher-KPP forward baseline. A paper-grade real-data model would still need reporting intensity, control actions, detection bias, forest or terrain masks, yearly policy changes, and spatially varying diffusion/reaction terms.